# Data Preprocessing

In [1]:
import os
from collections import defaultdict
import json
import torchaudio
import torchaudio.functional as F
import soundfile as sf
os.add_dll_directory(r'C:\ffmpeg\bin')

import torch
import torchcodec
# For class specific prompts
from tinytag import TinyTag

In [2]:
input_root = "./data"
output_root = "./data_preprocessed_general"
target_sr = 44100
target_channels = 2

class_mapping = defaultdict(int)
dropped = 0


for root, dirs, files in os.walk(input_root):
    head, class_name = os.path.split(root)

    # skips non class directories
    if not files:
        continue

    # create output dir for each class
    output_path = os.path.join(output_root, class_name)
    os.makedirs(output_path, exist_ok=True)

    # For hybrid prompt
    tag = TinyTag.get(os.path.join(root, files[0]))
    bird_title = tag.title if tag.title else class_name
    # prompt = f"Field recording of a {bird_title} singing in nature, stereo audio."

    # general prompt
    prompt = "A field recording of a bird singing in nature, stereo audio."

    # xeno canto has one extra class not contained in kaggle, so it may not be perfectly alphabetically
    if class_name not in class_mapping:
        class_mapping[class_name] = {"id" : len(class_mapping), "name" : bird_title}

    for file in files:
        if file.lower().endswith(".mp3"):
            # define paths
            in_file = os.path.join(root, file)
            filename = os.path.splitext(file)[0]
            out_file = os.path.join(output_path, filename)
            try:
                # sf also works on linux training server
                data, sr = sf.read(in_file)
                waveform = torch.from_numpy(data).float()
                if waveform.dim() == 1:
                    waveform = waveform.unsqueeze(0)
                else:
                    waveform = waveform.t()
                # resample
                if sr != target_sr:
                    waveform = F.resample(waveform, sr, target_sr)

                # convert channels
                num_channels = waveform.shape[0]
                if num_channels == 1:
                    waveform = waveform.repeat(target_channels, 1)
                elif num_channels > 2:
                    waveform = waveform[:2, :]

                # convert to wav
                torchaudio.save(out_file + ".wav", waveform, target_sr)
                
                duration_seconds = int(waveform.shape[1] / target_sr)
                # add json containing "prompt" and IntCondition
                cur_class_id = class_mapping[class_name]["id"]
                
                conditioning_dict = {
                    "prompt": prompt,
                    "species_id": cur_class_id,
                    "seconds_total": duration_seconds
                }
                json.dump(conditioning_dict, open(out_file + ".json", "w"))

            except Exception as e:
                dropped += 1
                print(f"Failed to process {in_file}, error: {e}")

with open("class_mapping.json", "w") as f:
    json.dump(class_mapping, f, indent=4)
print("Class mappings saved")
print("Files processed successfully. Samples dropped:", dropped)

Failed to process ./data\xenoCanto\amepip\XC113723.mp3, error: Unspecified internal error.
Failed to process ./data\xenoCanto\dusfly\XC356012.mp3, error: Unspecified internal error.
Class mappings saved
Files processed successfully. Samples dropped: 2
